In [ ]:
import datetime

from google.cloud import bigquery

In [ ]:
# Dataset parameters
bq_compute_project = "example-media-project"
project_id = "example-storage-project"
dataset_id = "user"

start_date = "2021-01-01"
end_date = "2021-12-31"
imp_thresh = 50
max_per_hashtag = 2000
n_no_hashtag = 10000000

In [ ]:
def human_format(num):
    num = float(f'{num:.3g}')
    magnitude = 0
    while abs(num) >= 1000:
        magnitude += 1
        num /= 1000.0
    return '{}{}'.format(f'{num:f}'.rstrip('0').rstrip('.'), ['', 'K', 'M', 'B', 'T'][magnitude])
  
table_id = f"text_image_pairs_s{human_format(max_per_hashtag)}_o{human_format(n_no_hashtag)}_not_protected_{start_date.replace('-', '')}_{end_date.replace('-', '')}"
print(table_id)

In [ ]:
query = f"""
WITH 
en_tweet_inventory AS (
    SELECT 
        tweet_id, author_id, tweet_date, tweet_text, hashtag, hashtags, media, num_impressions,
    FROM ( 
        SELECT 
            tweetId AS tweet_id, 
            userId AS author_id, 
            text AS tweet_text, 
            DATE(_PARTITIONTIME) AS tweet_date,
            media,
            split(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(regexp_replace(
                LOWER(ARRAY_TO_STRING(hashtags, ",")), '[-_.!?$~ー]', ''), "[àáâãäå]", 'a'), "æ", "ae"), "ç", "c"), "[èéêë]", "e") ,"[ìíîï]", "i"), "[òóôõö]", "o"), "œ", "oe"), "[ùúûü]", "u"), "[ýÿ]", "y"
            ), ",") hashtags,
        FROM `example-tweetsource-project.user.unhydrated_flat`
        WHERE DATE(_PARTITIONTIME) >= '{start_date}' AND DATE(_PARTITIONTIME) <= '{end_date}'
        AND language = "en"
    ) 
    CROSS JOIN UNNEST(hashtags) hashtag
    INNER JOIN (
       SELECT tid AS tweet_id, sum(impr) num_impressions
       FROM `example-bizinsights-project.user.tweet_impressions_engagements_per_day`
       WHERE DATE(_PARTITIONTIME) >= '{start_date}' AND DATE(_PARTITIONTIME) <= '{end_date}'
       AND impr >= {imp_thresh}
       GROUP BY 1
   ) # Filter tweets with impressions under threshold
   USING(tweet_id)   
), 
en_tweet_inventory_unnested_media AS (
    SELECT
        tweet_id, 
        author_id, 
        tweet_date, 
        tweet_text, 
        hashtag, 
        hashtags, 
        media_url_https, 
        media_each.nsfw AS media_nsfw,
        num_impressions,
    FROM en_tweet_inventory tweets
    CROSS JOIN UNNEST(media) media_each
    WHERE media_each.is_protected = False 
), # Unnest media and filter out protected images
selected_tweets AS(
    SELECT 
        tweet_id, 
        author_id,
        tweet_date, 
        tweet_text, 
        hashtag, 
        hashtags, 
        media_url_https,
        media_nsfw,
        num_impressions,
    FROM 
        (SELECT *, ROW_NUMBER() OVER (PARTITION BY hashtag ORDER BY RAND()) tweet_rank FROM en_tweet_inventory_unnested_media) 
    WHERE tweet_rank <= IF(hashtag = "", {n_no_hashtag}, {max_per_hashtag})
), # Select n_no_hashtag tweets without hashtags and at most max_per_hashtag tweets per hashtag (to try to make dataset more balanced)
selected_tweets_one_per_hashtag AS(
    SELECT 
        tweet_id, 
        author_id, 
        tweet_date, 
        tweet_text, 
        hashtags, 
        media_url_https, 
        media_nsfw,
        num_impressions
    FROM (
      SELECT 
        *, 
        ROW_NUMBER() OVER (PARTITION BY tweet_id, media_url_https) ht_rank
      FROM 
        selected_tweets
    )
    WHERE ht_rank = 1 
), # Eliminate duplicate tweets after unnesting hashtags; can still have same tweet with different media more than once
selected_tweets_one_per_hashtag_public AS (
  SELECT 
    t.*
  FROM
    selected_tweets_one_per_hashtag t
  INNER JOIN 
    (
      SELECT 
        DISTINCT id
      FROM 
        `example-usersource-project.user.usersource_flat`
      WHERE DATE(_PARTITIONTIME) >= '{start_date}' AND DATE(_PARTITIONTIME) <= '{end_date}'
      AND is_protected = FALSE
      AND deactivated = FALSE
    ) usersource
  ON t.author_id = usersource.id
)
SELECT 
    *, ROW_NUMBER() OVER (ORDER BY RAND()) rk # Shuffle and add row index
FROM 
    selected_tweets_one_per_hashtag_public
ORDER BY rk
"""

In [ ]:
# Construct a BigQuery client object.
client = bigquery.Client(project=bq_compute_project)

destination = f"{project_id}.{dataset_id}.{table_id}"
job_config = bigquery.QueryJobConfig(destination=destination)

# Start the query, passing in the extra configuration.
query_job = client.query(query, job_config=job_config)  # Make an API request.
query_job.result()  # Wait for the job to complete.

print(f"Query results loaded to the table {destination}")

In [ ]:
table_ref = bigquery.DatasetReference(project_id, dataset_id).table(table_id)
table = client.get_table(table_ref)

# Set table to expire 21 days from now
expiration = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=21)
table.expires = expiration
table = client.update_table(table, ["expires"])